# 1. Target: SalePrice

Continues from [0_load_and_orient.ipynb](0_load_and_orient.ipynb) — reloads
the same setup so this notebook runs standalone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import probplot, boxcox

In [ ]:
# load csv data for both train and test
df_train = pd.read_csv("../../data/train.csv")
df_test = pd.read_csv("../../data/test.csv")

In [ ]:
# Feature type categorization, from data_description.txt
# each list is sorted() so the final order is alphabetical regardless of
# how the source below groups them (kept grouped here for readability)

IDENTIFIER = ["Id"]
TARGET = ["SalePrice"]

NUMERIC_COLS = sorted([
    "LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF",
    "GrLivArea", "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",
    "BedroomAbvGr", "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces",
    "GarageCars", "GarageArea", "WoodDeckSF", "OpenPorchSF",
    "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal",
    # temporal — usually engineered into ages/deltas rather than used raw
    "YearBuilt", "YearRemodAdd", "GarageYrBlt", "MoSold", "YrSold",
])

# has a genuine order -> integer-encode in that order, don't one-hot
ORDINAL_COLS = sorted([
    "OverallQual", "OverallCond", "LotShape", "LandSlope",
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual",
    "Functional", "FireplaceQu", "GarageFinish", "GarageQual",
    "GarageCond", "PavedDrive", "PoolQC", "Utilities",
])

# no inherent order -> one-hot/dummy encode
NOMINAL_COLS = sorted([
    "MSSubClass",  # gotcha: stored as int, but it's a dwelling-type code
    "MSZoning", "Street", "Alley", "LandContour", "LotConfig",
    "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle",
    "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType",
    "Foundation", "Heating", "CentralAir", "Electrical", "GarageType",
    "MiscFeature", "SaleType", "SaleCondition", "Fence",
])

all_cols = NUMERIC_COLS + ORDINAL_COLS + NOMINAL_COLS
expected = set(df_train.columns) - set(IDENTIFIER) - set(TARGET)
assert set(all_cols) == expected, set(all_cols) ^ expected
assert len(all_cols) == len(expected)  # no duplicates
len(NUMERIC_COLS), len(ORDINAL_COLS), len(NOMINAL_COLS)

In [ ]:
df_train.SalePrice

In [ ]:
# check null values in target
df_train["SalePrice"].isnull().sum()

In [ ]:
# descriptive stats summary
df_train["SalePrice"].describe()

In [ ]:
# boxplot
sns.boxplot(x=df_train["SalePrice"], orient="h")

In [ ]:
# histogram
sns.displot(x = df_train["SalePrice"])

In [ ]:
sns.displot(x = df_train["SalePrice"], kind="kde")

In [ ]:
sns.histplot(x = df_train["SalePrice"], kde=True)

In [ ]:
# outlier range
q1 = df_train["SalePrice"].quantile(0.25)
q3 = df_train["SalePrice"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
lower, upper

In [ ]:
outliers = df_train[(df_train["SalePrice"] < lower) | (df_train["SalePrice"] > upper)]
outliers[["Id", "SalePrice"]]

In [ ]:
(outliers.Id).shape

In [ ]:
# median, mean, std, skewness, kurtosis
print("median:", df_train["SalePrice"].median())
print("mean:", df_train["SalePrice"].mean())
print("std:", df_train["SalePrice"].std())
print("skew:", df_train["SalePrice"].skew())
print("kurt:", df_train["SalePrice"].kurt())


In [ ]:
# QQ-Plot: raw SalePrice
probplot(df_train["SalePrice"], plot=plt)
plt.show()

In [ ]:
# log transform using log1p
log_sale_price = pd.Series(np.log1p(df_train["SalePrice"]))
log_sale_price.name = "log_sale_price"
log_sale_price

In [ ]:
# histplot on log_sale_price
sns.histplot(x = log_sale_price, kde=True, )

In [ ]:
# median, mean, std, skewness, kurtosis of log sale price
print("median:", log_sale_price.median())
print("mean:", log_sale_price.mean())
print("std:", log_sale_price.std())
print("skew:", log_sale_price.skew())
print("kurt:", log_sale_price.kurt())

In [ ]:
# QQ-Plot: log sale price
probplot(log_sale_price, plot=plt)
plt.show()

In [ ]:
sns.boxplot(x = log_sale_price, orient="h")

In [ ]:
# outliers in log sale price
q1 = log_sale_price.quantile(0.25)
q3 = log_sale_price.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = log_sale_price[(log_sale_price < lower) | (log_sale_price > upper)]
len(outliers)


In [ ]:
# left and right outliers count
low_outliers = log_sale_price[log_sale_price < lower]
high_outliers = log_sale_price[log_sale_price > upper]
len(low_outliers), len(high_outliers)


In [ ]:
# standard scaled sale price
scaled_sale_price = (df_train["SalePrice"] - df_train["SalePrice"].mean()) / df_train["SalePrice"].std()
scaled_sale_price

In [ ]:
# QQ-Plot: scaled sale price
probplot(scaled_sale_price, plot=plt)
plt.show()

In [ ]:
# histplot
sns.histplot(x = scaled_sale_price, kde=True)

In [ ]:
# QQ-Plot: scaled log sale price
scaled_log_sale_price= (log_sale_price - log_sale_price.mean()) / log_sale_price.std()
scaled_log_sale_price

In [ ]:
probplot(scaled_log_sale_price, plot=plt)
plt.show()

In [ ]:
print(f"Scaled log sale price kurtosis: {scaled_log_sale_price.kurt()}")

In [ ]:
df_train[NUMERIC_COLS].skew().sort_values(ascending=False)

In [ ]:
(df_train[NUMERIC_COLS] == 0).mean().sort_values(ascending=False)

In [ ]:
# scatter plot every numeric feature against the target, in a grid
ncols = 6
nrows = int(np.ceil(len(NUMERIC_COLS) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(24, 24))

for ax, col in zip(axes.flat, NUMERIC_COLS):
    sns.scatterplot(x=df_train[col], y=df_train["SalePrice"], ax=ax)
    ax.set_title(col)

# hide any unused grid slots (33 cols don't fill a 6x6=36 grid)
for ax in axes.flat[len(NUMERIC_COLS):]:
    ax.axis('off')

plt.tight_layout()


In [ ]:
# scatter plot every numeric feature against the (log) target, in a grid
ncols = 6
nrows = int(np.ceil(len(NUMERIC_COLS) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(24, 24))

for ax, col in zip(axes.flat, NUMERIC_COLS):
    sns.scatterplot(x=df_train[col], y=log_sale_price, ax=ax)
    ax.set_title(col)

# hide any unused grid slots (33 cols don't fill a 6x6=36 grid)
for ax in axes.flat[len(NUMERIC_COLS):]:
    ax.axis('off')

plt.tight_layout()

In [ ]:
xt, lam = boxcox(df_train["SalePrice"])
print(lam)

In [ ]:
xt

In [ ]:
probplot(xt, plot=plt)
plt.show()

### Decision: model on `log_sale_price`

Raw `SalePrice` is heavily right-skewed (skew 1.88, kurtosis 6.54), confirmed
visually by the QQ-plot's one-sided tail deviation — the right tail bows
sharply off the reference line while the left tail stays close to it.
`log1p(SalePrice)` is close to symmetric (skew 0.12, kurtosis 0.81) with a
QQ-plot that tracks the reference line except at the extreme tails.

Box-Cox was also checked as a comparison: the fitted λ ≈ -0.077, close to 0
(the log special case), so it doesn't meaningfully outperform a plain log
here and adds a fitted parameter to carry around for the inverse transform.

Going forward, `log_sale_price` is the modeling target. Predictions get
`expm1`-transformed back to raw dollars for evaluation/submission.